# Custom Dataset & DataLoader in PyTorch

**Interview Focus**: Data loading pipeline — Dataset, DataLoader, collate functions, samplers.

**Key Concepts**:
- Map-style vs Iterable-style Dataset
- Custom collate functions for variable-length sequences
- Custom samplers (weighted, distributed)
- Efficient data loading patterns for NLP

In [ ]:
import torch
from torch.utils.data import (
    Dataset, IterableDataset, DataLoader,
    Sampler, WeightedRandomSampler, BatchSampler,
    SequentialSampler, RandomSampler
)
from torch.nn.utils.rnn import pad_sequence
import numpy as np
from collections import Counter
import math
import random

## 1. Map-Style Dataset (`__getitem__` + `__len__`)

The standard Dataset. Supports random access via index.

**When to use**: Data fits in memory or is on local disk with fast random access.

In [ ]:
class TextClassificationDataset(Dataset):
    """Map-style dataset for text classification with variable-length sequences."""
    
    def __init__(self, texts, labels, vocab=None, max_len=None):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
        
        # Build vocabulary if not provided
        if vocab is None:
            self.vocab = self._build_vocab(texts)
        else:
            self.vocab = vocab
    
    def _build_vocab(self, texts, min_freq=1):
        counter = Counter()
        for text in texts:
            counter.update(text.lower().split())
        
        vocab = {'<pad>': 0, '<unk>': 1}
        for word, count in counter.items():
            if count >= min_freq:
                vocab[word] = len(vocab)
        return vocab
    
    def _tokenize(self, text):
        tokens = text.lower().split()
        ids = [self.vocab.get(t, self.vocab['<unk>']) for t in tokens]
        if self.max_len:
            ids = ids[:self.max_len]
        return ids
    
    def __getitem__(self, idx):
        """Return (token_ids_tensor, label) for a single example."""
        token_ids = self._tokenize(self.texts[idx])
        return torch.tensor(token_ids, dtype=torch.long), self.labels[idx]
    
    def __len__(self):
        return len(self.texts)


# Demo data — variable-length sentences
texts = [
    "this movie is great",
    "terrible film",
    "i loved every minute of this wonderful movie",
    "bad",
    "excellent acting and directing",
    "worst movie i have ever seen in my entire life honestly",
    "good",
    "the plot was boring and predictable",
]
labels = [1, 0, 1, 0, 1, 0, 1, 0]

dataset = TextClassificationDataset(texts, labels)
print(f"Dataset size: {len(dataset)}")
print(f"Vocab size: {len(dataset.vocab)}")

# Access single items
token_ids, label = dataset[0]
print(f"\nSample 0: tokens={token_ids}, label={label}")
token_ids, label = dataset[2]
print(f"Sample 2: tokens={token_ids}, label={label}")
print("Note: different lengths — we need collation for batching.")

## 2. Iterable-Style Dataset (`__iter__`)

Streams data. No random access, no `__len__`.

**When to use**: Streaming from a database/API, data too large for random access, reading from multiple sharded files.

In [ ]:
class StreamingTextDataset(IterableDataset):
    """Iterable dataset that simulates streaming from a data source."""
    
    def __init__(self, file_paths=None, data_chunks=None, shuffle_chunks=True):
        """In production: file_paths would be shard files.
        Here we simulate with data_chunks."""
        self.data_chunks = data_chunks or []
        self.shuffle_chunks = shuffle_chunks
    
    def _process_chunk(self, chunk):
        """Process a single chunk of data (simulate file reading)."""
        for item in chunk:
            # Simulate tokenization
            tokens = torch.tensor([ord(c) % 50 for c in item], dtype=torch.long)
            yield tokens
    
    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        chunks = list(self.data_chunks)
        
        if self.shuffle_chunks:
            random.shuffle(chunks)
        
        # If using multiple workers, split chunks across workers
        if worker_info is not None:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            chunks = [c for i, c in enumerate(chunks) if i % num_workers == worker_id]
        
        for chunk in chunks:
            yield from self._process_chunk(chunk)


# Demo
chunk1 = ["hello world", "foo bar"]
chunk2 = ["streaming data", "is useful"]
chunk3 = ["for large datasets", "that dont fit"]

streaming_ds = StreamingTextDataset(data_chunks=[chunk1, chunk2, chunk3])

print("Streaming through all items:")
for i, item in enumerate(streaming_ds):
    print(f"  Item {i}: shape={item.shape}")

## 3. Custom Collate Function

The collate function defines how individual samples are combined into a batch. Critical for variable-length sequences.

**Default collate**: stacks tensors — fails if sizes differ.  
**Custom collate**: pad to max length in batch, create attention masks, etc.

In [ ]:
def collate_fn_pad(batch):
    """Collate variable-length sequences by padding to the longest in the batch.
    
    Args:
        batch: list of (token_ids_tensor, label) tuples
    Returns:
        padded_sequences: (batch_size, max_seq_len)
        attention_mask:   (batch_size, max_seq_len)  -- 1 for real tokens, 0 for padding
        labels:           (batch_size,)
        lengths:          (batch_size,)  -- original lengths
    """
    token_ids_list, labels = zip(*batch)
    
    # Record original lengths
    lengths = torch.tensor([len(ids) for ids in token_ids_list])
    
    # Pad sequences to max length in this batch (pad_sequence pads with 0 by default)
    padded = pad_sequence(token_ids_list, batch_first=True, padding_value=0)
    
    # Create attention mask
    attention_mask = torch.zeros_like(padded)
    for i, length in enumerate(lengths):
        attention_mask[i, :length] = 1
    
    labels = torch.tensor(labels, dtype=torch.long)
    
    return padded, attention_mask, labels, lengths


# Demo: show padding in action
loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_fn_pad)

for batch_idx, (padded, mask, labels, lengths) in enumerate(loader):
    print(f"Batch {batch_idx}:")
    print(f"  Padded shape: {padded.shape}")
    print(f"  Lengths:      {lengths.tolist()}")
    print(f"  Labels:       {labels.tolist()}")
    print(f"  Attention mask row 0: {mask[0].tolist()}")
    print(f"  Padded tokens row 0: {padded[0].tolist()}")
    print()

## 4. Advanced Collate: Bucketing by Length

Padding all sequences to the same length wastes compute. Bucketing groups similar-length sequences together to minimize padding.

In [ ]:
class BucketBatchSampler(Sampler):
    """Sampler that groups examples by length to minimize padding.
    
    1. Sort indices by sequence length
    2. Create buckets of batch_size
    3. Optionally shuffle the bucket order (but not within buckets)
    """
    
    def __init__(self, lengths, batch_size, shuffle=True, drop_last=False):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last
    
    def __iter__(self):
        # Sort indices by length
        sorted_indices = sorted(range(len(self.lengths)), key=lambda i: self.lengths[i])
        
        # Create batches from sorted indices
        batches = []
        for i in range(0, len(sorted_indices), self.batch_size):
            batch = sorted_indices[i:i + self.batch_size]
            if self.drop_last and len(batch) < self.batch_size:
                continue
            batches.append(batch)
        
        # Shuffle batch order to add randomness
        if self.shuffle:
            random.shuffle(batches)
        
        for batch in batches:
            yield batch
    
    def __len__(self):
        if self.drop_last:
            return len(self.lengths) // self.batch_size
        return math.ceil(len(self.lengths) / self.batch_size)


# Get lengths from our dataset
seq_lengths = [len(dataset[i][0]) for i in range(len(dataset))]
print("Sequence lengths:", seq_lengths)

bucket_sampler = BucketBatchSampler(seq_lengths, batch_size=3, shuffle=False)

loader_bucketed = DataLoader(
    dataset,
    batch_sampler=bucket_sampler,
    collate_fn=collate_fn_pad
)

print("\nBucketed batches (sorted by length):")
for batch_idx, (padded, mask, labels, lengths) in enumerate(loader_bucketed):
    padding_tokens = (padded == 0).sum().item()
    total_tokens = padded.numel()
    print(f"  Batch {batch_idx}: lengths={lengths.tolist()}, "
          f"padded_shape={padded.shape}, "
          f"padding_ratio={padding_tokens/total_tokens:.1%}")

## 5. Custom Samplers

### Weighted Sampling
For imbalanced datasets — oversample the minority class.

In [ ]:
class CustomWeightedSampler(Sampler):
    """Weighted random sampler from scratch."""
    
    def __init__(self, labels, num_samples=None, replacement=True):
        self.labels = labels
        self.num_samples = num_samples or len(labels)
        self.replacement = replacement
        
        # Compute weights: inverse of class frequency
        class_counts = Counter(labels)
        total = len(labels)
        class_weights = {cls: total / count for cls, count in class_counts.items()}
        self.weights = torch.tensor([class_weights[label] for label in labels],
                                     dtype=torch.double)
    
    def __iter__(self):
        indices = torch.multinomial(self.weights, self.num_samples,
                                    replacement=self.replacement)
        return iter(indices.tolist())
    
    def __len__(self):
        return self.num_samples


# Demo: imbalanced dataset (90% class 0, 10% class 1)
imbalanced_labels = [0]*90 + [1]*10
sampler = CustomWeightedSampler(imbalanced_labels, num_samples=100)

# Count how often each class is sampled
sampled_labels = [imbalanced_labels[i] for i in sampler]
counts = Counter(sampled_labels)
print(f"Original distribution: {Counter(imbalanced_labels)}")
print(f"After weighted sampling: {counts}")
print("Roughly balanced now.")

### Distributed Sampling (Concept)

In distributed training, each GPU processes a different subset. The sampler partitions data across ranks.

In [ ]:
class SimpleDistributedSampler(Sampler):
    """Minimal distributed sampler — demonstrates the partitioning logic.
    
    Each rank gets a non-overlapping subset of indices.
    Pads to make all ranks have the same number of samples.
    """
    
    def __init__(self, dataset_size, num_replicas, rank, shuffle=True, seed=42):
        self.dataset_size = dataset_size
        self.num_replicas = num_replicas  # total number of GPUs
        self.rank = rank                  # this GPU's ID
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        
        # Pad so every rank gets the same number of samples
        self.total_size = math.ceil(dataset_size / num_replicas) * num_replicas
        self.num_samples = self.total_size // num_replicas
    
    def set_epoch(self, epoch):
        """Must call this each epoch so shuffling is different."""
        self.epoch = epoch
    
    def __iter__(self):
        # Deterministic shuffling based on epoch
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch)
        
        if self.shuffle:
            indices = torch.randperm(self.dataset_size, generator=g).tolist()
        else:
            indices = list(range(self.dataset_size))
        
        # Pad to total_size by wrapping around
        padding = self.total_size - len(indices)
        indices += indices[:padding]
        
        # Subsample for this rank
        indices = indices[self.rank::self.num_replicas]
        assert len(indices) == self.num_samples
        
        return iter(indices)
    
    def __len__(self):
        return self.num_samples


# Demo: 10 samples, 3 GPUs
for rank in range(3):
    sampler = SimpleDistributedSampler(dataset_size=10, num_replicas=3, rank=rank, shuffle=False)
    indices = list(sampler)
    print(f"  Rank {rank}: indices = {indices} (count={len(indices)})")

print("\nNote: rank 2 gets a repeated sample (index 0) to keep counts equal.")

## 6. Full DataLoader with All Options

In [ ]:
# Standard DataLoader with our custom pieces

# Option 1: Basic — shuffle, custom collate
loader_basic = DataLoader(
    dataset,
    batch_size=3,
    shuffle=True,            # uses RandomSampler internally
    collate_fn=collate_fn_pad,
    num_workers=0,           # 0 = main process (use >0 for real training)
    pin_memory=False,        # True for GPU training
    drop_last=False,
)

# Option 2: Weighted sampler (mutually exclusive with shuffle=True)
weighted_sampler = WeightedRandomSampler(
    weights=[1.0] * len(dataset),
    num_samples=len(dataset),
    replacement=True
)
loader_weighted = DataLoader(
    dataset,
    batch_size=3,
    sampler=weighted_sampler,  # cannot use shuffle= with sampler=
    collate_fn=collate_fn_pad,
)

# Option 3: Bucket batch sampler
loader_bucket = DataLoader(
    dataset,
    batch_sampler=BucketBatchSampler(seq_lengths, batch_size=3),
    collate_fn=collate_fn_pad,
)

print("Basic loader batches:")
for padded, mask, labels, lengths in loader_basic:
    print(f"  shape={padded.shape}, lengths={lengths.tolist()}")

## 7. Multi-Worker Data Loading

Key points:
- `num_workers > 0` spawns separate processes for data loading
- Workers prefetch batches while the GPU trains on the current batch
- `pin_memory=True` copies tensors to CUDA pinned memory for faster GPU transfer
- For IterableDataset, **you must handle worker splitting** in `__iter__`

In [ ]:
def worker_init_fn(worker_id):
    """Ensure each worker has a different random seed.
    Without this, all workers produce identical random augmentations.
    """
    np.random.seed(np.random.get_state()[1][0] + worker_id)

# In production:
# loader = DataLoader(
#     dataset,
#     batch_size=32,
#     num_workers=4,
#     pin_memory=True,       # for GPU
#     prefetch_factor=2,     # how many batches each worker prefetches
#     persistent_workers=True,  # keep workers alive between epochs
#     worker_init_fn=worker_init_fn,
#     collate_fn=collate_fn_pad,
# )

print("Key DataLoader parameters:")
print("  num_workers=0    : load in main process (debugging)")
print("  num_workers=4    : 4 prefetch workers")
print("  pin_memory=True  : faster CPU -> GPU transfer")
print("  prefetch_factor=2: each worker prefetches 2 batches")
print("  persistent_workers=True: don't respawn workers each epoch")

## 8. End-to-End: Training with Custom DataLoader

In [ ]:
import torch.nn as nn

class SimpleTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, input_ids, attention_mask):
        # (batch, seq_len, embed_dim)
        embeds = self.embedding(input_ids)
        # Masked mean pooling
        mask_expanded = attention_mask.unsqueeze(-1).float()  # (batch, seq_len, 1)
        summed = (embeds * mask_expanded).sum(dim=1)          # (batch, embed_dim)
        counts = mask_expanded.sum(dim=1).clamp(min=1)        # (batch, 1)
        pooled = summed / counts                               # (batch, embed_dim)
        return self.fc(pooled)


# Training loop
model = SimpleTextClassifier(vocab_size=len(dataset.vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn_pad)

for epoch in range(5):
    total_loss = 0
    for padded, mask, labels, lengths in train_loader:
        logits = model(padded, mask)
        loss = criterion(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}: loss={total_loss:.4f}")

## Interview Questions & Answers

---

**Q: `__getitem__` vs `__iter__`? When to use IterableDataset?**

| | Map-style (`__getitem__`) | Iterable (`__iter__`) |
|---|---|---|
| Access | Random access by index | Sequential only |
| Shuffling | DataLoader handles it | Must implement yourself |
| Multi-worker | Automatic partitioning | Must partition in `__iter__` |
| Use when | Data on disk/in memory | Streaming, databases, huge datasets |
| `__len__` | Required | Optional |

Use IterableDataset when: data comes from a stream (API, database cursor), dataset is too large to index, or reading from multiple sharded files.

---

**Q: How does distributed sampling work?**

1. DistributedSampler creates a deterministic permutation (same seed on all ranks)
2. Partitions indices: rank k gets indices `[k, k+N, k+2N, ...]`
3. Pads dataset so all ranks get exactly the same number of samples
4. Must call `sampler.set_epoch(epoch)` each epoch so shuffling varies
5. Gradients are synchronized via AllReduce after backward pass

---

**Q: Why use a custom collate function?**

Default collate calls `torch.stack` which requires all tensors to have the same shape. For NLP with variable-length sequences, you need to:
- Pad to the max length in the batch
- Create attention masks
- Optionally sort by length for `pack_padded_sequence`

---

**Q: `num_workers` best practices?**

- Start with `num_workers = 4 * num_gpus`
- Too many → CPU contention, diminishing returns
- `pin_memory=True` when training on GPU
- `persistent_workers=True` avoids respawning overhead each epoch
- For debugging, use `num_workers=0`

## Quick Reference

```python
# Standard pattern
dataset = MyDataset(data)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=my_collate,
    drop_last=True,  # for training, avoid small last batch
)

# Distributed training pattern
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=32, sampler=sampler)
for epoch in range(n_epochs):
    sampler.set_epoch(epoch)
    for batch in loader:
        ...
```